In [1]:
# vizualize the features of sources

import numpy as np
import pandas as pd
from matplotlib import pyplot
from matplotlib.patches import Rectangle
#import matplotlib.colors as mcolors
from matplotlib import cm
plt = pyplot

import umap
from sklearn.preprocessing import StandardScaler
import seaborn as sns


import sys
if '../lib/' not in sys.path:
    sys.path.append('../lib/')
import dio3
import multi_fgl_utils as util
import plot_util as putil

In [2]:
def plot_colortable(colors, names, title, ncols=2):

    cell_width = 400
    cell_height = 30
    swatch_width = 150
    margin = 12
    topmargin = 40

    n = len(names)
    #ncols = 4 - emptycols
    nrows = n // ncols + int(n % ncols > 0)

    width = cell_width * ncols + 2 * margin
    height = cell_height * nrows + margin + topmargin
    dpi = 72

    fig, ax = plt.subplots(figsize=(width / dpi, height / dpi), dpi=dpi)
    fig.subplots_adjust(margin/width, margin/height,
                        (width-margin)/width, (height-topmargin)/height)
    ax.set_xlim(0, cell_width * 4)
    ax.set_ylim(cell_height * (nrows-0.5), -cell_height/2.)
    ax.yaxis.set_visible(False)
    ax.xaxis.set_visible(False)
    ax.set_axis_off()
    ax.set_title(title, fontsize=24, loc="left", pad=10)

    for i, name in enumerate(names):
        row = i % nrows
        col = i // nrows
        y = row * cell_height

        swatch_start_x = cell_width * col
        text_pos_x = cell_width * col + swatch_width + 7

        ax.text(text_pos_x, y, name, fontsize=14,
                horizontalalignment='left',
                verticalalignment='center')

        ax.add_patch(
            Rectangle(xy=(swatch_start_x, y-9), width=swatch_width,
                      height=18, facecolor=colors[i], edgecolor='0.7')
        )

    return fig


def key2x(key):
    ff = 0.5
    ef = np.array(list(key), dtype=int)
    nn = np.arange(0, len(ef))
    return np.sum(ff**(ef * nn) * np.sign(ef))

def keys2colors(keys, cmap='brg'):
    xs = np.array([key2x(key) for key in keys])
    xs -= np.min(xs)
    xs /= np.max(xs)
    return [cm.get_cmap(cmap)(x) for x in xs]


In [5]:
# load the cat tree
folder = '../data_out/'
#tree_cat_fn = folder + 'tree_cat.yaml'
#xdf_fn = folder + 'xdf.csv'

mode = 'no_bcu_spp' # 'all_assoc', 'no_bcu_spp'
if mode == 'all_assoc':
    mode_str = ''
else:
    mode_str = '_%s' % mode

sep_model = 'RF' # 'GMM', 'RF'
if sep_model == 'RF':
    sep_str = '_RF'
else:
    sep_str = ''
nmin = 60

#out_fn = folder + 'cat_tree_nmin%i%s.yaml' % (nmin, sep_str)
tree_cat_fn = folder + 'tree_cat_nmin%i%s%s.yaml' % (nmin, sep_str, mode_str)
Eind = 1000
xdf_fn = folder_out + 'xdf_fgl_Eind%iMeV.csv' % Eind
class_name = 'CLASS1'



tree_cat_dict = dio3.loaddict(tree_cat_fn)
xdf = pd.read_csv(xdf_fn, index_col=0)
features0 = [name for name in xdf.columns if name != 'class']
cols = ['sin_lat', 'log_signif', 'lp_index1000', 'lp_beta', 'lp_signif_curv', 
        'log_var_index', class_name]
xdf = xdf[cols]
features = [name for name in xdf.columns if name != class_name]
unas_mask = (xdf['class'] == 'unas') | (xdf['class'] == 'unk')
xdf = xdf[~unas_mask]
ns = len(xdf)
putil.setup_figure_pars()
#print(features0)
#print(xdf[features])

0

In [6]:
tree_cat_dict

{'000': ['bll', 'sbg', 'rdg'],
 '0010': ['msp'],
 '0011': ['snr', 'pwn', 'gc', 'gal'],
 '010': ['ssrq', 'sey', 'css', 'agn', 'fsrq', 'nlsy1'],
 '011': ['psr', 'glc', 'sfr', 'bin', 'hmb', 'nov', 'lmb']}

In [7]:
if 0:
    cat_tree = list(tree_cat_dict.keys())
    cat_forest = util.get_forest(max_depth=4, max_tree=cat_tree)
    print(len(cat_forest))
    ind = min(len(cat_forest), 15)
    subtree = cat_forest[ind]
    cut_subtree = util.sub_tree2tree_cat_dict(subtree, cat_tree, tree_cat_dict)
else:
    cut_subtree = tree_cat_dict

In [9]:
xdf.columns

Index(['sin_lat', 'log_signif', 'lp_index500', 'lp_beta', 'lp_signif_curv',
       'log_var_index', 'class'],
      dtype='object')

In [8]:
ys = np.zeros(ns, dtype='<U6')
for key, cls in cut_subtree.items():
    cls_mask = util.get_cl_mask(cls, xdf)
    ys[cls_mask] = key

KeyError: 'CLASS1'

In [ ]:
scale_data = False
if scale_data:
    scaled_data = StandardScaler().fit_transform(xdf[features][~unas_mask])
else:
    scaled_data = np.array(xdf[features][~unas_mask])
reducer = umap.UMAP()
embedding = reducer.fit_transform(scaled_data)

In [ ]:
classes = list(cut_subtree.keys())
cmap = 'brg' # 'tab20', 'rainbow'
colors = keys2colors(classes, cmap=cmap)
cls2color_dict = dict(zip(classes, colors)) #{key:key2color(key) for key in classes}
ps_colors = [cls2color_dict[yy] for yy in ys]

In [ ]:
cls2color_dict

In [ ]:
title = 'Classes'
class_colors = [cls2color_dict[cls] for cls in classes]
fgl_cats = [cut_subtree[cls] for cls in classes]
fig = plot_colortable(class_colors, fgl_cats, title, ncols=1)
fig = plot_colortable(class_colors, classes, title)

In [ ]:
pyplot.scatter(embedding[:, 0], embedding[:, 1], s=5, c=ps_colors)